# 01 - Data Cleaning and Merging
The purpose of this notebook is to load, clean, and, merge the raw workout and bodyweight datasets. The primary steps include standardizing data types, handling missing values, and combining the two datasets on the "Date" column. The final output is a clean, master DataFrame that will be used for subsequent in-depth analysis and visualization.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)  # display all columns
pd.set_option('display.max_rows', 150)  # display up to 150 rows
pd.set_option('display.max_colwidth', None)  # increase column width
pd.set_option('display.float_format', '{:.2f}'.format)  # format floats to 2 decimal places 

In [2]:
# loading csv files
workouts_df = pd.read_csv('../data/raw/workout_data.csv')
weight_df = pd.read_csv('../data/raw/measurement_data.csv')

### 2.1 - Previewing the Workout Data
To get a better idea of the way this data is organized, let's view the head along with any attributes with null values. This will help me decide what data I can safely delete.

In [3]:
workouts_df.head()  # preview the data

,title,start_time,end_time,description,exercise_title,superset_id,exercise_notes,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
0,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Single Leg Press (Machine),0.00,egym lowkey feel lighter,0,normal,50.00,8.00,NaN,NaN,8.50
1,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Single Leg Press (Machine),0.00,egym lowkey feel lighter,1,normal,80.00,4.00,NaN,NaN,10.00
2,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Leg Press Horizontal (Machine),0.00,egym,0,normal,200.00,8.00,NaN,NaN,8.00
3,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Leg Press Horizontal (Machine),0.00,egym,1,normal,250.00,8.00,NaN,NaN,9.00
4,Legs,"2 Sep 2025, 21:53","2 Sep 2025, 22:40",it was hella busy so we gave up like a nafis workout,Seated Leg Curl (Machine),NaN,NaN,0,warmup,50.00,8.00,NaN,NaN,6.00


In [4]:
workouts_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2104 entries, 0 to 2103
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             2104 non-null   object 
 1   start_time        2104 non-null   object 
 2   end_time          2104 non-null   object 
 3   description       314 non-null    object 
 4   exercise_title    2104 non-null   object 
 5   superset_id       78 non-null     float64
 6   exercise_notes    130 non-null    object 
 7   set_index         2104 non-null   int64  
 8   set_type          2104 non-null   object 
 9   weight_lbs        1957 non-null   float64
 10  reps              2042 non-null   float64
 11  distance_miles    11 non-null     float64
 12  duration_seconds  62 non-null     float64
 13  rpe               2004 non-null   float64
dtypes: float64(6), int64(1), object(7)
memory usage: 230.3+ KB


In [5]:
workouts_df.isnull().sum()  # check how many null values are present for each column

title                  0
start_time             0
end_time               0
description         1790
exercise_title         0
superset_id         2026
exercise_notes      1974
set_index              0
set_type               0
weight_lbs           147
reps                  62
distance_miles      2093
duration_seconds    2042
rpe                  100
dtype: int64

### 2.2 - Removing Workout Metadata
We can immediately see some metadata that will not be necessary in our data analysis, so we can remove them. This includes the description and exercise notes. The title of the workout itself is also not strictly necessary in the data analysis, but may make it easier to organize specific exercises done on each day into their respective muscle groups.

In [6]:
workouts_df = workouts_df.drop(['description', 'exercise_notes'], axis=1)  # remove the listed columns

### 2.3 - Verifying Irrelevant Exercise Data
Since we aim to look at strength training data, we can get rid of any data related to cardio or other exercise types. These exercises will not have weight or rep information. However, some exercises, like pull ups, may not have weight information since it is based on bodyweight. These exercises are vital for checking trends in strength so they should not be deleted. Thus, we will analyze the data closer to verify what data should and should not be deleted.

In [7]:
workouts_df[workouts_df['weight_lbs'].isnull()]  # select the rows without any value for weight_lbs, but without permanently deleting the rest from the df

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
7,Legs,"31 Aug 2025, 17:40","31 Aug 2025, 18:33",Spinning,NaN,0,normal,NaN,NaN,NaN,311.00,NaN
18,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
19,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Chin Up,NaN,0,normal,NaN,12.00,NaN,NaN,8.50
20,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Chin Up,NaN,1,normal,NaN,8.00,NaN,NaN,9.50
41,Legs,"22 Aug 2025, 16:17","22 Aug 2025, 16:49",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
52,Chest/Back,"19 Aug 2025, 18:11","19 Aug 2025, 18:55",Pull Up,NaN,0,normal,NaN,10.00,NaN,NaN,9.00
53,Chest/Back,"19 Aug 2025, 18:11","19 Aug 2025, 18:55",Pull Up,NaN,1,normal,NaN,7.00,NaN,NaN,10.00
63,early bird gets the milkshake,"16 Aug 2025, 14:25","16 Aug 2025, 15:47",Chin Up,NaN,0,normal,NaN,12.00,NaN,NaN,9.50
64,early bird gets the milkshake,"16 Aug 2025, 14:25","16 Aug 2025, 15:47",Chin Up,NaN,1,normal,NaN,7.00,NaN,NaN,9.50
98,Arms,"7 Aug 2025, 15:04","7 Aug 2025, 16:36",Chin Up,NaN,0,normal,NaN,12.00,NaN,NaN,9.00


In [8]:
workouts_df[workouts_df['reps'].isnull()]

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
7,Legs,"31 Aug 2025, 17:40","31 Aug 2025, 18:33",Spinning,NaN,0,normal,NaN,NaN,NaN,311.00,NaN
18,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
41,Legs,"22 Aug 2025, 16:17","22 Aug 2025, 16:49",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
136,Arms,"2 Aug 2025, 15:38","2 Aug 2025, 17:08",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
165,Arms,"30 Jul 2025, 15:06","30 Jul 2025, 16:13",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
180,Chest/Back,"27 Jul 2025, 16:35","27 Jul 2025, 17:49",Spinning,NaN,0,normal,NaN,NaN,NaN,180.00,NaN
208,chesticles or back tickles?,"24 Jul 2025, 14:57","24 Jul 2025, 16:04",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
260,Arms,"14 Jul 2025, 15:29","14 Jul 2025, 16:31",Spinning,NaN,0,normal,NaN,NaN,NaN,510.00,NaN
272,Chest/Back,"8 Jul 2025, 15:39","8 Jul 2025, 16:15",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
295,little bit of this little bit of that nasar chowdhury,"25 Jun 2025, 16:19","25 Jun 2025, 16:59",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN


### 2.4 - Deleting Irrelevant Exercise Data
Since I have verified that bodyweight exercises do not have weight information, I will remove exercises based on other attributes, like distance and duration. Then, we can check back and make sure that there are no remaining irrelevant exercises in the dataset and look at the full datasets data types using info().

In [9]:
workouts_df[workouts_df['distance_miles'].notna()]  # select only the rows with a value for distance_miles, without permanently deleting the rest

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
1486,Core/Cardio,"27 Nov 2024, 08:03","27 Nov 2024, 09:18",Treadmill,NaN,0,normal,NaN,NaN,0.52,1074.00,NaN
1514,Afternoon workout 💪,"25 Nov 2024, 16:44","25 Nov 2024, 17:19",Treadmill,NaN,0,normal,NaN,NaN,0.75,1250.00,NaN
1594,Core/Cardio,"21 Nov 2024, 07:31","21 Nov 2024, 08:45",Treadmill,NaN,0,normal,NaN,NaN,0.97,1505.00,NaN
1638,Evening workout 🏋️,"18 Nov 2024, 17:57","18 Nov 2024, 19:01",Treadmill,NaN,0,normal,NaN,NaN,1.74,2710.00,NaN
1675,Afternoon workout 💪,"16 Nov 2024, 16:44","16 Nov 2024, 18:05",Treadmill,NaN,0,normal,NaN,NaN,1.51,2400.00,NaN
1692,Evening workout 🏋️,"15 Nov 2024, 19:26","15 Nov 2024, 20:22",Treadmill,NaN,0,normal,NaN,NaN,1.15,1815.00,NaN
1737,Chest/Back,"14 Nov 2024, 10:07","14 Nov 2024, 11:46",Treadmill,NaN,0,normal,NaN,NaN,0.20,365.00,NaN
1753,Core/Cardio,"13 Nov 2024, 10:51","13 Nov 2024, 12:02",Treadmill,NaN,0,normal,NaN,NaN,0.82,1280.00,NaN
1900,Core/Cardio,"30 Oct 2024, 12:56","30 Oct 2024, 14:01",Treadmill,NaN,0,normal,NaN,NaN,0.25,572.00,NaN
2040,Core/Cardio,"7 Oct 2024, 19:58","7 Oct 2024, 20:54",Treadmill,NaN,0,normal,NaN,NaN,0.32,900.00,NaN


In [10]:
workouts_df[workouts_df['duration_seconds'].notna()]

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe
7,Legs,"31 Aug 2025, 17:40","31 Aug 2025, 18:33",Spinning,NaN,0,normal,NaN,NaN,NaN,311.00,NaN
18,Arms,"28 Aug 2025, 12:42","28 Aug 2025, 13:30",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
41,Legs,"22 Aug 2025, 16:17","22 Aug 2025, 16:49",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
136,Arms,"2 Aug 2025, 15:38","2 Aug 2025, 17:08",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
165,Arms,"30 Jul 2025, 15:06","30 Jul 2025, 16:13",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
180,Chest/Back,"27 Jul 2025, 16:35","27 Jul 2025, 17:49",Spinning,NaN,0,normal,NaN,NaN,NaN,180.00,NaN
208,chesticles or back tickles?,"24 Jul 2025, 14:57","24 Jul 2025, 16:04",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
260,Arms,"14 Jul 2025, 15:29","14 Jul 2025, 16:31",Spinning,NaN,0,normal,NaN,NaN,NaN,510.00,NaN
272,Chest/Back,"8 Jul 2025, 15:39","8 Jul 2025, 16:15",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN
295,little bit of this little bit of that nasar chowdhury,"25 Jun 2025, 16:19","25 Jun 2025, 16:59",Spinning,NaN,0,normal,NaN,NaN,NaN,300.00,NaN


In [11]:
# update the df to only keep rows where distance_miles and duration_seconds are null
workouts_df = workouts_df[workouts_df['distance_miles'].isnull()]
workouts_df = workouts_df[workouts_df['duration_seconds'].isnull()]

In [12]:
workouts_df[workouts_df['reps'].isnull()]

,title,start_time,end_time,exercise_title,superset_id,set_index,set_type,weight_lbs,reps,distance_miles,duration_seconds,rpe


In [13]:
workouts_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2042 entries, 0 to 2103
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title             2042 non-null   object 
 1   start_time        2042 non-null   object 
 2   end_time          2042 non-null   object 
 3   exercise_title    2042 non-null   object 
 4   superset_id       78 non-null     float64
 5   set_index         2042 non-null   int64  
 6   set_type          2042 non-null   object 
 7   weight_lbs        1957 non-null   float64
 8   reps              2042 non-null   float64
 9   distance_miles    0 non-null      float64
 10  duration_seconds  0 non-null      float64
 11  rpe               2004 non-null   float64
dtypes: float64(6), int64(1), object(5)
memory usage: 207.4+ KB


### 2.5 - Final Removal of Irrelevant Workout Information
Since all our cardio workouts have been removed from the dataset, we can also remove the columns that track distance and duration. Additionally, since it is not particularly pertinent to this analysis we can also get rid of superset information.

In [14]:
workouts_df = workouts_df.drop(['superset_id', 'distance_miles', 'duration_seconds'], axis=1)

In [15]:
workouts_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2042 entries, 0 to 2103
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   title           2042 non-null   object 
 1   start_time      2042 non-null   object 
 2   end_time        2042 non-null   object 
 3   exercise_title  2042 non-null   object 
 4   set_index       2042 non-null   int64  
 5   set_type        2042 non-null   object 
 6   weight_lbs      1957 non-null   float64
 7   reps            2042 non-null   float64
 8   rpe             2004 non-null   float64
dtypes: float64(3), int64(1), object(5)
memory usage: 159.5+ KB


### 2.6 - Converting Workout Date Information to Datetime
To ensure that merging the datasets is a smooth process, we need to ensure that both date columns are datetime objects. However, there is a start time and an end time, something that will not be present on bodyweight data. We can keep and convert both to datetime and then later determine the workout duration if necessary for our analysis. But to ensure a common column for both datasets, we can create a new column called date based on the start_time, then convert that to datetime as well. We will also bring the date attribute to the front of the dataset, right after title, for better readability. Then we can use info() to ensure the correct data types are being used. Then we can use info() to ensure the correct data types are being used.

In [16]:
workouts_df['start_time'] = pd.to_datetime(workouts_df['start_time'])
workouts_df['end_time'] = pd.to_datetime(workouts_df['end_time'])

In [17]:
workouts_df['date'] = workouts_df['start_time'].dt.date
workouts_df['date'] = pd.to_datetime(workouts_df['date'])

In [18]:
cols = workouts_df.columns.tolist()
cols.insert(1, cols.pop(cols.index('date')))
workouts_df = workouts_df[cols]

In [19]:
workouts_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2042 entries, 0 to 2103
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   title           2042 non-null   object        
 1   date            2042 non-null   datetime64[ns]
 2   start_time      2042 non-null   datetime64[ns]
 3   end_time        2042 non-null   datetime64[ns]
 4   exercise_title  2042 non-null   object        
 5   set_index       2042 non-null   int64         
 6   set_type        2042 non-null   object        
 7   weight_lbs      1957 non-null   float64       
 8   reps            2042 non-null   float64       
 9   rpe             2004 non-null   float64       
dtypes: datetime64[ns](3), float64(3), int64(1), object(3)
memory usage: 175.5+ KB


### 2.7 - Previewing the Bodyweight Data
Now we will do much of the same data processing on the bodyweight dataset that we just did to the workout dataset. Then, after cleaning this dataset, we will be ready to merge the two datasets into the master DataFrame to be used for analysis.

In [20]:
weight_df.head()

,date,weight_lbs,fat_percent,neck_in,shoulder_in,chest_in,left_bicep_in,right_bicep_in,left_forearm_in,right_forearm_in,abdomen_in,waist_in,hips_in,left_thigh_in,right_thigh_in,left_calf_in,right_calf_in
0,"24 Jun 2020, 00:00",111.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"13 Jun 2022, 00:00",114.99,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,"9 Apr 2023, 00:00",119.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,"10 Apr 2023, 00:00",118.61,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,"11 Apr 2023, 00:00",119.01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
weight_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 763 entries, 0 to 762
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   date              763 non-null    object 
 1   weight_lbs        763 non-null    float64
 2   fat_percent       0 non-null      float64
 3   neck_in           0 non-null      float64
 4   shoulder_in       0 non-null      float64
 5   chest_in          0 non-null      float64
 6   left_bicep_in     0 non-null      float64
 7   right_bicep_in    0 non-null      float64
 8   left_forearm_in   0 non-null      float64
 9   right_forearm_in  0 non-null      float64
 10  abdomen_in        0 non-null      float64
 11  waist_in          0 non-null      float64
 12  hips_in           0 non-null      float64
 13  left_thigh_in     0 non-null      float64
 14  right_thigh_in    0 non-null      float64
 15  left_calf_in      0 non-null      float64
 16  right_calf_in     0 non-null      float64
dt

In [22]:
weight_df.isnull().sum()

date                  0
weight_lbs            0
fat_percent         763
neck_in             763
shoulder_in         763
chest_in            763
left_bicep_in       763
right_bicep_in      763
left_forearm_in     763
right_forearm_in    763
abdomen_in          763
waist_in            763
hips_in             763
left_thigh_in       763
right_thigh_in      763
left_calf_in        763
right_calf_in       763
dtype: int64

### 2.8 - Removing Empty Measurement Columns
As we can see, most of the columns are fully empty since I only tracked my bodyweight and the date. Therefore, we can remove all the other columns, leaving only what we need.

In [23]:
weight_df = weight_df[['date', 'weight_lbs']]

In [24]:
weight_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 763 entries, 0 to 762
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        763 non-null    object 
 1   weight_lbs  763 non-null    float64
dtypes: float64(1), object(1)
memory usage: 12.1+ KB


### 2.9 - Converting Bodyweight Date Information to Datetime
Just like we did for the workout data, we convert the dates of when I weighed myself to be in datetime format so that it is consistent when we merge the datasets. Then, we verify the operation was successful using info().

In [25]:
weight_df['date'] = pd.to_datetime(weight_df['date'])

In [26]:
weight_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 763 entries, 0 to 762
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   date        763 non-null    datetime64[ns]
 1   weight_lbs  763 non-null    float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 12.1 KB


### 2.10 - Merging Datasets
Since both datasets are adequately cleaned and standardized, now we can merge them using the common date attribute and verify that all the data looks as expected. We will also explicitly sort the new dataset to ensure that the order is ascending to view change over time, starting from the beginning.

In [27]:
final_df = pd.merge(workouts_df, weight_df, on='date', how='outer')  # keep all data from workouts_df and weight_df but combine the date attribute

In [28]:
final_df.sort_values(by='date', ascending=True, inplace=True)  # sort in ascending order

In [29]:
final_df.head()

,title,date,start_time,end_time,exercise_title,set_index,set_type,weight_lbs_x,reps,rpe,weight_lbs_y
0,NaN,2020-06-24,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,111.99
1,NaN,2022-06-13,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,114.99
2,NaN,2023-04-09,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,119.01
3,NaN,2023-04-10,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,118.61
4,NaN,2023-04-11,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,119.01


In [30]:
final_df.tail()

,title,date,start_time,end_time,exercise_title,set_index,set_type,weight_lbs_x,reps,rpe,weight_lbs_y
2685,NaN,2025-09-10,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,139.71
2686,NaN,2025-09-11,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,139.31
2687,NaN,2025-09-12,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,136.80
2688,NaN,2025-09-13,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,137.90
2689,NaN,2025-09-14,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,138.49


In [31]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2690 entries, 0 to 2689
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   title           2042 non-null   object        
 1   date            2690 non-null   datetime64[ns]
 2   start_time      2042 non-null   datetime64[ns]
 3   end_time        2042 non-null   datetime64[ns]
 4   exercise_title  2042 non-null   object        
 5   set_index       2042 non-null   float64       
 6   set_type        2042 non-null   object        
 7   weight_lbs_x    1957 non-null   float64       
 8   reps            2042 non-null   float64       
 9   rpe             2004 non-null   float64       
 10  weight_lbs_y    2573 non-null   float64       
dtypes: datetime64[ns](3), float64(5), object(3)
memory usage: 252.2+ KB


In [32]:
final_df[final_df['reps'].notna()]  # checking for workout data since head and tail are all bodyweight data

,title,date,start_time,end_time,exercise_title,set_index,set_type,weight_lbs_x,reps,rpe,weight_lbs_y
471,Arms,2024-10-05,2024-10-05 15:40:00,2024-10-05 16:40:00,Triceps Extension (Dumbbell),2.00,normal,20.00,8.00,NaN,127.60
470,Arms,2024-10-05,2024-10-05 15:40:00,2024-10-05 16:40:00,Triceps Extension (Dumbbell),1.00,normal,20.00,8.00,NaN,127.60
469,Arms,2024-10-05,2024-10-05 15:40:00,2024-10-05 16:40:00,Triceps Extension (Dumbbell),0.00,normal,20.00,8.00,NaN,127.60
468,Arms,2024-10-05,2024-10-05 15:40:00,2024-10-05 16:40:00,Hammer Curl (Dumbbell),2.00,normal,30.00,8.00,NaN,127.60
467,Arms,2024-10-05,2024-10-05 15:40:00,2024-10-05 16:40:00,Hammer Curl (Dumbbell),1.00,normal,30.00,8.00,NaN,127.60
...,...,...,...,...,...,...,...,...,...,...,...
2676,Legs,2025-09-02,2025-09-02 21:53:00,2025-09-02 22:40:00,Leg Press Horizontal (Machine),1.00,normal,250.00,8.00,9.00,136.71
2678,Legs,2025-09-02,2025-09-02 21:53:00,2025-09-02 22:40:00,Seated Leg Curl (Machine),1.00,normal,75.00,8.00,9.00,136.71
2674,Legs,2025-09-02,2025-09-02 21:53:00,2025-09-02 22:40:00,Single Leg Press (Machine),1.00,normal,80.00,4.00,10.00,136.71
2673,Legs,2025-09-02,2025-09-02 21:53:00,2025-09-02 22:40:00,Single Leg Press (Machine),0.00,normal,50.00,8.00,8.50,136.71


### 2.11 - Fixing Issues
As we can see, there is one key issue with the data. We had two columns called weight_lbs which led to the merge renaming these to weight_lbs_x and weight_lbs_y, respectively. We will now rename these for clarity and rename all other potentially unclear column names. Additionally, since the date is what links our datasets, we will move this attribute to the front, and the weight_lbs_y column, which is our bodyweight to be right after it. This will make the data be presented more logically.

In [33]:
final_df = final_df.rename(columns={
    'title': 'workout_title',
    'weight_lbs_x': 'working_weight_lbs',
    'weight_lbs_y': 'bodyweight_lbs'
})

In [34]:
cols = final_df.columns.tolist()
cols.insert(0, cols.pop(cols.index('date')))
final_df = final_df[cols]
cols.insert(1, cols.pop(cols.index('bodyweight_lbs')))
final_df = final_df[cols]

In [35]:
final_df.head()

,date,bodyweight_lbs,workout_title,start_time,end_time,exercise_title,set_index,set_type,working_weight_lbs,reps,rpe
0,2020-06-24,111.99,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN
1,2022-06-13,114.99,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN
2,2023-04-09,119.01,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-04-10,118.61,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN
4,2023-04-11,119.01,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN


### Saving the Final DataFrame
Now that we have our merged datasets and any issues have been resolved, we are ready to save it to a new file so that we can explore and analyze our master dataset in our next notebook.

In [36]:
final_df.to_csv('../data/processed/processed_data.csv', index=False)  # index=False to prevent saving df index as a separate column

### 2.12 - Conclusion and Next Steps
In this notebook, the raw workout and bodyweight datasets were loaded, cleaned, and processed. Key steps included standardizing the 'date' column for merging, handling missing values, and removing data irrelevant to the analysis. Then, we merged the two datasets into a single, unified DataFrame and sorted chronologically. This master dataset has been saved to `data/processed/cleaned_fitness_data.csv`. Now, we will explore and analyze the data in our next notebook.